# Phase 2: Exploratory Data Analysis (EDA)
### Student Dropout & Performance Prediction System
This notebook contains the 8 core visualizations required for analyzing student performance and predicting dropout risk.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

In [ ]:
# Load data from SQLite database
db_path = "../data/students.db"
if not os.path.exists(db_path):
    db_path = "data/students.db" # Fallback if run from project root

conn = sqlite3.connect(db_path)
df = pd.read_sql("SELECT * FROM students", conn)
conn.close()
print(f"Loaded dataset with {len(df)} records and {len(df.columns)} features.")

### Plot 1: Target Class Distribution
Shows the exact counts of Graduates, Enrolled, and Dropout students.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Target', data=df, palette='viridis')
plt.title('Target Class Distribution')
plt.xlabel('Student Outcome')
plt.ylabel('Count')
plt.savefig('../assets/screenshots/plot1_class_distribution.png', bbox_inches='tight', dpi=150) if os.path.exists('../assets') else plt.savefig('plot1_class_distribution.png')
plt.show()

### Plot 2: Missing Values Heatmap
Snapshot of data completeness and quality.

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values Snapshot (Data Quality Heatmap)')
plt.show()

### Plot 3: Age Distribution by Outcome
Do older students show a higher propensity to drop out?

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='Target', y='Age at enrollment', data=df, palette='magma')
plt.title('Age Distribution by Outcome')
plt.xlabel('Student Status')
plt.ylabel('Age at Enrollment')
plt.show()

### Plot 4: Scholarship Holder vs. Dropout Status
Socioeconomic support and its relationship with educational success.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Scholarship holder', hue='Target', data=df, palette='coolwarm')
plt.title('Impact of Scholarship Holder Status on Student Outcome')
plt.xlabel('Scholarship Holder (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

### Plot 5: Curricular Units Semester Grade Correlation
Comparing academic marks in 1st vs. 2nd semester.

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x='Curricular units 1st sem (grade)', 
    y='Curricular units 2nd sem (grade)', 
    hue='Target', 
    data=df, 
    alpha=0.6, 
    palette='Set1'
)
plt.title('Grade Correlation (1st Sem vs 2nd Sem)')
plt.xlabel('1st Semester Grade Average')
plt.ylabel('2nd Semester Grade Average')
plt.show()

### Plot 6: Correlation Matrix Heatmap
Feature relationships across numeric features.

In [ ]:
numeric_cols = [
    'Age at enrollment', 'Admission grade', 
    'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)',
    'Unemployment rate', 'Inflation rate', 'GDP'
]
corr = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.show()

### Plot 7: Macroeconomic GDP vs. Dropout Rate
Analyzing macro conditions against outcomes.

In [ ]:
gdp_df = df.groupby('GDP')['Target'].value_counts(normalize=True).unstack().fillna(0)
gdp_df['Dropout_Rate'] = gdp_df['Dropout'] * 100
gdp_df = gdp_df.sort_index()

plt.figure(figsize=(8, 5))
plt.plot(gdp_df.index, gdp_df['Dropout_Rate'], marker='o', linewidth=2, color='crimson')
plt.title('Student Dropout Rate vs GDP Growth')
plt.xlabel('GDP Growth Rate (%)')
plt.ylabel('Dropout Rate (%)')
plt.grid(True)
plt.show()

### Plot 8: Feature Importance (Post-Model training)
Top 10 predictors derived from the best performing machine learning model.

In [ ]:
fi_path = "../models/feature_importance.pkl"
if not os.path.exists(fi_path):
    fi_path = "models/feature_importance.pkl"

if os.path.exists(fi_path):
    sorted_fi = joblib.load(fi_path)
    top_10 = sorted_fi[:10]
    features = [x[0] for x in top_10]
    importances = [x[1] for x in top_10]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances, y=features, palette='rocket')
    plt.title('Top 10 Predictor Features')
    plt.xlabel('Importance Value')
    plt.ylabel('Feature')
    plt.savefig('../assets/screenshots/plot8_feature_importance.png', bbox_inches='tight', dpi=150) if os.path.exists('../assets') else plt.savefig('plot8_feature_importance.png')
    plt.show()
else:
    print("Feature importance pickle not found. Run model training (src/train.py) first to generate visual!")